In [ ]:
from qiskit import QuantumCircuit, transpile, QuantumRegister, ClassicalRegister
from qiskit.result import marginal_counts, marginal_distribution

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeFez, FakeTorino

from qiskit.visualization import plot_histogram

import os
import pickle
import numpy as np
import networkx as nx

# [[5,1,3]] Code Transport via Entanglement Swapping

This notebook implements quantum state transport using entanglement swapping instead of direct SWAP chains.

## Protocol Overview

For a path A → B → C → D:

1. **Bell Pair Distribution**: Create noisy Bell pairs on each edge
2. **Entanglement Swapping**: At intermediate nodes, perform Bell measurements to extend entanglement
3. **Teleportation**: Bell measurement at source teleports encoded state to sink
4. **Syndrome Measurement**: Extract syndrome at sink (no corrections applied)

# Constants and Configuration

In [ ]:
MAX_HOPS = 3
MIN_HOPS = 2
NUM_SHOTS = 4096
OPTIMIZATION_LEVEL = 0

SKIP_EXISTING_PATHS = False
NETWORK_GRAPH = '../../pickle_files/2x3_grid_diag_network_graph.pkl'

# Physical Qubit Mapping

## Layout:
- **Node qubits**: 9 per node (5 data + 4 ancilla)
- **Edge qubits**: 10 per edge (5 for left node + 5 for right node) for Bell pairs

In [ ]:
# Each network node has 9 qubits:
#   - Qubits 0-4: Data qubits (for encoding at source)
#   - Qubits 5-8: Ancilla qubits for syndrome measurement

NODE_PHYSICAL_QUBITS = {
    0: list(range(0, 9)),      # Node 0 → physical qubits 0-8
    1: list(range(9, 18)),     # Node 1 → physical qubits 9-17
    2: list(range(18, 27)),    # Node 2 → physical qubits 18-26
    3: list(range(27, 36)),    # Node 3 → physical qubits 27-35
    4: list(range(36, 45)),    # Node 4 → physical qubits 36-44
    5: list(range(45, 54)),    # Node 5 → physical qubits 45-53
}

# Edge qubits for Bell pairs: 10 qubits per edge
# For edge (A, B): qubits[0:5] are A's halves, qubits[5:10] are B's halves
EDGE_PHYSICAL_QUBITS = {
    (0, 1): list(range(54, 64)),
    (0, 3): list(range(64, 74)),
    (1, 2): list(range(74, 84)),
    (1, 4): list(range(84, 94)),
    (2, 5): list(range(94, 104)),
    (3, 4): list(range(104, 114)),
    (4, 5): list(range(114, 124)),
    (0, 4): list(range(124, 134)),
    (1, 5): list(range(134, 144)),
    (2, 4): list(range(144, 154)),
    (1, 3): list(range(154, 164)),
}

# Path qubits for entanglement distribution (1 per edge, used sequentially for 5 pairs)
# These model the physical channel between nodes
PATH_PHYSICAL_QUBITS = {
    (0, 1): 164,
    (0, 3): 165,
    (1, 2): 166,
    (1, 4): 167,
    (2, 5): 168,
    (3, 4): 169,
    (4, 5): 170,
    (0, 4): 171,
    (1, 5): 172,
    (2, 4): 173,
    (1, 3): 174,
}

# Total physical qubits: 54 (nodes) + 110 (edge Bell pairs) + 11 (path channels) = 175
TOTAL_QUBITS = (len(NODE_PHYSICAL_QUBITS) * 9 +
               sum(len(q) for q in EDGE_PHYSICAL_QUBITS.values()) +
               len(PATH_PHYSICAL_QUBITS))

print("Node to Physical Qubit Mapping:")
for node, qubits in NODE_PHYSICAL_QUBITS.items():
    print(f"  Node {node}: physical qubits {qubits[0]}-{qubits[-1]}")

print(f"\nEdge Bell Pair Qubits (10 per edge):")
for edge, qubits in EDGE_PHYSICAL_QUBITS.items():
    print(f"  Edge {edge}: qubits {qubits[0]}-{qubits[-1]} (A's half: {qubits[0]}-{qubits[4]}, B's half: {qubits[5]}-{qubits[9]})")

print(f"\nPath Qubits (channel for distribution):")
for edge, qubit in PATH_PHYSICAL_QUBITS.items():
    print(f"  Edge {edge}: path qubit {qubit}")

print(f"\nTotal physical qubits used: {TOTAL_QUBITS}")

# Noise Model Setup

In [ ]:
fake_backend = FakeFez()
network_channel_noise = NoiseModel.from_backend(fake_backend)

FEZ_BASIS_GATES = ['cz', 'id', 'rz', 'sx', 'x']

print(f"Noise Model extracted from {fake_backend}")
print(f"  Backend qubits: {fake_backend.num_qubits}")
print(f"  Noise model basis gates: {network_channel_noise.basis_gates}")
print(f"\nCircuits must be transpiled to basis gates: {FEZ_BASIS_GATES}")

# =============================================================================
# SIMULATION METHOD NOTES
# =============================================================================
# For 175 qubits, statevector is impossible (2^175 amplitudes).
# 
# Best options:
#   - 'matrix_product_state' (MPS): Good for circuits with limited entanglement.
#     This is our best choice - handles large qubit counts efficiently.
#   - 'tensor_network': Similar to MPS, experimental.
#   - 'stabilizer': Extremely fast but Clifford-only (no noise).
#
# Speed tips:
#   - device='GPU' if CUDA available
#   - Lower optimization_level in transpile (0 or 1)
#   - Batch multiple circuits with run([circuit_list])
#
print(f"\\nSimulation method: matrix_product_state (best for {TOTAL_QUBITS} qubits)")

# Load Network Graph

In [ ]:
with open(NETWORK_GRAPH, 'rb') as f:
    network_graph = pickle.load(f)

nx.draw(network_graph, with_labels=True)

In [ ]:
from itertools import combinations

ALL_HOP_PATHS = []
for pairs in combinations(network_graph.nodes, 2):
    multi_hop_paths = nx.all_simple_paths(network_graph, source=pairs[0], target=pairs[1], cutoff=MAX_HOPS)
    ALL_HOP_PATHS.extend([x for x in multi_hop_paths if len(x) > MIN_HOPS])

print(f"Total paths to process: {len(ALL_HOP_PATHS)}")
print(f"Sample paths: {ALL_HOP_PATHS[:5]}")

# [[5,1,3]] Code Helpers

Stabilizers: XZZXI, IXZZX, XIXZZ, ZXIXZ

In [ ]:
STABILIZERS = ["XZZXI", "IXZZX", "XIXZZ", "ZXIXZ"]


def apply_encoding(qc, qubits):
    """
    Apply [[5,1,3]] encoding circuit.
    
    Encodes a single logical qubit (on qubit[4]) into the 5-qubit code.
    Qubits 0-3 should be initialized to |0⟩ before encoding.
    
    Args:
        qc: QuantumCircuit to add gates to
        qubits: List of 5 qubit indices [q0, q1, q2, q3, q4] where q4 is the data qubit
    """
    qc.h(qubits[0])
    qc.z(qubits[0])
    qc.cz(qubits[0], qubits[4])
    qc.cx(qubits[0], qubits[4])

    qc.h(qubits[1])
    qc.cx(qubits[1], qubits[4])

    qc.h(qubits[2])
    qc.cx(qubits[2], qubits[4])
    qc.cz(qubits[2], qubits[1])
    qc.cz(qubits[2], qubits[0])

    qc.h(qubits[3])
    qc.z(qubits[3])
    qc.cz(qubits[3], qubits[4])
    qc.cx(qubits[3], qubits[4])
    qc.cz(qubits[3], qubits[2])
    qc.cz(qubits[3], qubits[0])


def apply_syndrome_measurement(qc, data_qubits, ancilla_qubits, classical_bits):
    """
    Extract syndrome and measure ancilla qubits.
    
    Args:
        qc: QuantumCircuit to add gates to
        data_qubits: List of 5 data qubit indices [q0, q1, q2, q3, q4]
        ancilla_qubits: List of 4 ancilla qubit indices [a0, a1, a2, a3]
        classical_bits: ClassicalRegister or list of 4 classical bit indices
    """
    for idx, stabilizer in enumerate(STABILIZERS):
        ancilla = ancilla_qubits[idx]
        qc.h(ancilla)
        
        for qubit_idx, pauli in enumerate(stabilizer):
            if pauli == 'X':
                qc.cx(ancilla, data_qubits[qubit_idx])
            elif pauli == 'Z':
                qc.cz(ancilla, data_qubits[qubit_idx])
        
        qc.h(ancilla)
        qc.measure(ancilla, classical_bits[idx])

# Entanglement Swapping Primitives

In [ ]:
def get_edge_key(node_a, node_b):
    """Get canonical edge key (smaller node first)."""
    return (min(node_a, node_b), max(node_a, node_b))


def distribute_bell_pairs(qc, node_a, node_b):
    """
    Create and distribute 5 Bell pairs between two adjacent nodes.
    
    This models realistic entanglement distribution:
    1. Create Bell pair locally at node_a (between A's edge qubits and path qubit)
    2. SWAP the path qubit to node_b's edge qubits
    
    The SWAP through the path qubit introduces channel noise.
    
    Args:
        qc: QuantumCircuit
        node_a: Source node for Bell pair creation
        node_b: Destination node for the distributed half
    
    After this operation:
        - A's edge qubits (for this edge) are entangled with B's edge qubits
    """
    edge_key = get_edge_key(node_a, node_b)
    edge_qubits = EDGE_PHYSICAL_QUBITS[edge_key]
    path_qubit = PATH_PHYSICAL_QUBITS[edge_key]
    
    # Determine which half belongs to which node
    # Convention: qubits[0:5] belong to smaller node, qubits[5:10] to larger
    if node_a < node_b:
        a_qubits = edge_qubits[0:5]   # A's halves (stay at A)
        b_qubits = edge_qubits[5:10]  # B's halves (distributed to B)
    else:
        a_qubits = edge_qubits[5:10]
        b_qubits = edge_qubits[0:5]
    
    # Distribute 5 Bell pairs sequentially through the path qubit
    for i in range(5):
        # Step 1: Create Bell pair locally - H on A's qubit, CNOT to path qubit
        qc.h(a_qubits[i])
        qc.cx(a_qubits[i], path_qubit)
        
        # Step 2: SWAP path qubit to B's qubit (this is the noisy channel!)
        qc.swap(path_qubit, b_qubits[i])
        
        # Now a_qubits[i] is entangled with b_qubits[i]
        # The path_qubit is back to |0⟩ and ready for next pair


def apply_bell_measurement(qc, qubits_a, qubits_b, classical_bits):
    """
    Perform Bell measurements on pairs of qubits.
    
    Bell measurement: CNOT(a→b), H(a), then measure both.
    Results encode which Bell state was measured:
        00 → |Φ+⟩, 01 → |Ψ+⟩, 10 → |Φ-⟩, 11 → |Ψ-⟩
    
    Args:
        qc: QuantumCircuit
        qubits_a: List of 5 qubit indices (control qubits)
        qubits_b: List of 5 qubit indices (target qubits)
        classical_bits: List/register of 10 classical bits [a0..a4, b0..b4]
    """
    for i, (qa, qb) in enumerate(zip(qubits_a, qubits_b)):
        qc.cx(qa, qb)
        qc.h(qa)
        qc.measure(qa, classical_bits[i])      # First 5 bits for qubits_a
        qc.measure(qb, classical_bits[i + 5])  # Next 5 bits for qubits_b

# Main Circuit Builder: Entanglement Swapping Protocol

In [ ]:
def get_node_edge_qubits(node, neighbor):
    """
    Get the 5 edge qubits that belong to 'node' for the edge (node, neighbor).
    
    Convention: In EDGE_PHYSICAL_QUBITS[(min, max)]:
        - qubits[0:5] belong to the smaller node
        - qubits[5:10] belong to the larger node
    """
    edge_key = get_edge_key(node, neighbor)
    edge_qubits = EDGE_PHYSICAL_QUBITS[edge_key]
    
    if node < neighbor:
        return edge_qubits[0:5]
    else:
        return edge_qubits[5:10]


def build_entanglement_swapping_circuit(path, initial_state='0'):
    """
    Build circuit for [[5,1,3]] code transport via entanglement swapping.
    
    Protocol:
        1. Encode logical qubit at source into 5 physical qubits
        2. Distribute noisy Bell pairs along each edge (via SWAP through path qubits)
        3. Entanglement swapping at intermediate nodes (Bell measurements)
        4. Teleportation: Bell measurement at source between code qubits and link qubits
        5. Syndrome measurement at sink on the received qubits
    
    Args:
        path: List of node IDs, e.g., [0, 1, 2] for A→B→C
        initial_state: '0' or '1' for the logical qubit
    
    Returns:
        QuantumCircuit with all operations
    """
    print(f"Building entanglement swapping circuit for path: {path}")
    
    source = path[0]
    sink = path[-1]
    intermediate_nodes = path[1:-1]
    edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    
    print(f"  Source: {source}, Sink: {sink}, Intermediates: {intermediate_nodes}")
    print(f"  Edges: {edges}")
    
    # Create circuit
    qc = QuantumCircuit(TOTAL_QUBITS)
    
    # Classical registers:
    # - For each intermediate node: 10 bits for Bell measurement (entanglement swapping)
    # - For source: 10 bits for teleportation Bell measurement
    # - For sink: 4 bits for syndrome measurement
    
    swap_registers = {}
    for node in intermediate_nodes:
        reg = ClassicalRegister(10, name=f'c_swap_{node}')
        qc.add_register(reg)
        swap_registers[node] = reg
    
    teleport_register = ClassicalRegister(10, name=f'c_teleport_{source}')
    qc.add_register(teleport_register)
    
    syndrome_register = ClassicalRegister(4, name=f'c_syndrome_{sink}')
    qc.add_register(syndrome_register)
    
    # Get qubit assignments
    source_data_qubits = NODE_PHYSICAL_QUBITS[source][:5]  # Encoding qubits
    sink_ancilla_qubits = NODE_PHYSICAL_QUBITS[sink][5:9]  # Syndrome ancillas
    
    # =========================================================================
    # STEP 1: Encode logical qubit at source
    # =========================================================================
    if initial_state == '1':
        qc.x(source_data_qubits[4])  # Data qubit is index 4
    
    apply_encoding(qc, source_data_qubits)
    qc.barrier(label='Encoding Complete')
    
    # =========================================================================
    # STEP 2: Distribute Bell pairs along each edge (with channel noise via SWAP)
    # =========================================================================
    for edge in edges:
        distribute_bell_pairs(qc, edge[0], edge[1])
    
    qc.barrier(label='Bell Pairs Distributed')
    
    # =========================================================================
    # STEP 3: Entanglement swapping at intermediate nodes
    # =========================================================================
    # At each intermediate node B between A and C:
    #   - B has edge qubits connected to A (incoming)
    #   - B has edge qubits connected to C (outgoing)
    #   - Bell measurement on (incoming, outgoing) extends entanglement A↔C
    
    for i, node in enumerate(intermediate_nodes):
        prev_node = path[i]      # Node before this intermediate
        next_node = path[i + 2]  # Node after this intermediate
        
        # Get this node's qubits from each adjacent edge
        incoming_qubits = get_node_edge_qubits(node, prev_node)
        outgoing_qubits = get_node_edge_qubits(node, next_node)
        
        print(f"  Swapping at node {node}: incoming (from {prev_node})={incoming_qubits}, outgoing (to {next_node})={outgoing_qubits}")
        
        # Bell measurement to swap entanglement
        apply_bell_measurement(qc, incoming_qubits, outgoing_qubits, swap_registers[node])
    
    if intermediate_nodes:
        qc.barrier(label='Entanglement Swapping Complete')
    
    # =========================================================================
    # STEP 4: Teleportation from source to sink
    # =========================================================================
    # After entanglement swapping:
    #   - Source's edge qubits (to first neighbor) are now entangled with
    #   - Sink's edge qubits (from last neighbor)
    # 
    # Teleport by Bell measurement between encoded qubits and source's link qubits
    
    first_neighbor = path[1]
    source_link_qubits = get_node_edge_qubits(source, first_neighbor)
    
    print(f"  Teleportation: source_data={source_data_qubits}, source_link={source_link_qubits}")
    
    # Bell measurement between encoded qubits and source's link qubits
    apply_bell_measurement(qc, source_data_qubits, source_link_qubits, teleport_register)
    qc.barrier(label='Teleportation Complete')
    
    # =========================================================================
    # STEP 5: Syndrome measurement at sink
    # =========================================================================
    # The teleported state is now on sink's edge qubits (connected to last neighbor)
    
    last_neighbor = path[-2]
    sink_data_qubits = get_node_edge_qubits(sink, last_neighbor)
    
    print(f"  Syndrome measurement: sink_data={sink_data_qubits}, sink_ancilla={sink_ancilla_qubits}")
    
    apply_syndrome_measurement(qc, sink_data_qubits, sink_ancilla_qubits, syndrome_register)
    
    return qc

# Test Circuit Construction

In [ ]:
# Test with a simple path
test_path = [0, 1, 2]
test_circuit = build_entanglement_swapping_circuit(test_path, initial_state='0')

print(f"\nCircuit created with {test_circuit.num_qubits} qubits, {test_circuit.num_clbits} classical bits")
print(f"Classical registers: {[reg.name for reg in test_circuit.cregs]}")
print(f"Gate counts: {test_circuit.count_ops()}")

In [ ]:
test_circuit.draw('mpl', fold=-1, idle_wires=False)

# Noiseless Simulation Test

In [ ]:
# Test noiseless simulation
simulator = AerSimulator(noise_model=None, method='matrix_product_state')
decomposed_circuit = transpile(test_circuit, basis_gates=FEZ_BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL)

print(f"Decomposed circuit gate counts: {decomposed_circuit.count_ops()}")

result = simulator.run(decomposed_circuit, shots=NUM_SHOTS).result()

# Get syndrome register indices
syndrome_reg = test_circuit.cregs[-1]  # Last register is syndrome
syndrome_indices = [test_circuit.find_bit(bit).index for bit in syndrome_reg]
print(f"Syndrome register: {syndrome_reg.name}, bit indices: {syndrome_indices}")

# Extract syndrome counts
syndrome_counts = marginal_distribution(result.get_counts(), indices=syndrome_indices)
print(f"Syndrome counts (noiseless): {dict(sorted(syndrome_counts.items()))}")

In [ ]:
plot_histogram(syndrome_counts)

# Noisy Simulation Test

In [ ]:
# Noisy simulation
noisy_simulator = AerSimulator(
    noise_model=network_channel_noise,
    method='matrix_product_state'
)

noisy_result = noisy_simulator.run(decomposed_circuit, shots=NUM_SHOTS).result()
noisy_syndrome_counts = marginal_distribution(noisy_result.get_counts(), indices=syndrome_indices)

print(f"Syndrome counts (noisy): {dict(sorted(noisy_syndrome_counts.items()))}")

In [ ]:
plot_histogram(noisy_syndrome_counts)

# Data Generation

In [ ]:
class TimeAwareMeasurement:
    def __init__(self, path_edges, histogram, duration, latency_stats=None, code_type='513_ES'):
        """
        :param path_edges: List of tuples representing the path, e.g., [(0,1), (1,2)]
        :param histogram: A 16-dimensional numpy array of syndrome counts.
        :param duration: The duration or average latency of this measurement (seconds).
        :param latency_stats: A dictionary with additional stats.
        :param code_type: String indicating the code type, e.g., '513_ES' for entanglement swapping.
        """
        self.path_edges = path_edges
        self.histogram = histogram
        self.duration = duration
        self.latency_stats = latency_stats
        self.code_type = code_type

In [ ]:
TIME_AWARE_MEASUREMENTS = []

# Create noisy simulator
NOISY_AER_SIM = AerSimulator(
    noise_model=network_channel_noise,
    method='matrix_product_state'
)

print("=" * 70)
print("513 ENTANGLEMENT SWAPPING SYNDROME DATA GENERATION")
print("=" * 70)
print(f"Simulator: AerSimulator with FakeFez noise model")
print(f"Basis gates for decomposition: {FEZ_BASIS_GATES}")
print(f"Shots per circuit: {NUM_SHOTS}")
print(f"Total paths to process: {len(ALL_HOP_PATHS)}")
print("=" * 70)

for path_idx, path in enumerate(ALL_HOP_PATHS):
    print(f"\n[{path_idx + 1}/{len(ALL_HOP_PATHS)}] Processing path: {path}")
    
    # Build circuit
    circuit = build_entanglement_swapping_circuit(path, initial_state='0')
    
    # Get syndrome register indices
    syndrome_reg = circuit.cregs[-1]  # Last register is syndrome
    syndrome_indices = [circuit.find_bit(bit).index for bit in syndrome_reg]
    
    # Transpile and run
    decomposed = transpile(circuit, basis_gates=FEZ_BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL)
    result = NOISY_AER_SIM.run(decomposed, shots=NUM_SHOTS).result()
    
    # Extract syndrome histogram
    syndrome_counts = marginal_distribution(result.get_counts(), indices=syndrome_indices)
    
    # Build 16-element histogram
    histogram = np.zeros(16, dtype=np.float32)
    for bitstring, count in syndrome_counts.items():
        syndrome_val = int(bitstring, 2)
        histogram[syndrome_val] = count
    
    # Convert path to edges
    path_edges = [(path[i], path[i + 1]) for i in range(len(path) - 1)]
    
    print(f"  Path edges: {path_edges}")
    print(f"  Histogram: {histogram}")
    print(f"  No-error fraction: {histogram[0] / histogram.sum():.4f}")
    
    measurement = TimeAwareMeasurement(
        path_edges=path_edges,
        histogram=histogram,
        duration=0.0,
        latency_stats={'total_shots': NUM_SHOTS},
        code_type='513_ES'
    )
    TIME_AWARE_MEASUREMENTS.append(measurement)

print(f"\n{'=' * 70}")
print(f"Total measurements collected: {len(TIME_AWARE_MEASUREMENTS)}")
print(f"{'=' * 70}")

# Save Data

In [ ]:
DATA_DIR = '../../pickle_files/syndrome_data_513_es/'
os.makedirs(DATA_DIR, exist_ok=True)

with open(os.path.join(DATA_DIR, 'measurements.pkl'), 'wb') as f:
    pickle.dump(TIME_AWARE_MEASUREMENTS, f)

with open(os.path.join(DATA_DIR, 'graph.pkl'), 'wb') as f:
    pickle.dump(network_graph, f)

print(f"Data saved to {DATA_DIR}")